In [40]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open("names.txt").read().splitlines()

In [19]:
#dictionary of bigram counts
b = {}
for w in words:
    chs = ["<S>"] + list(w) + ["<E>"]
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [23]:
sortedB = sorted(b.items(), key = lambda kv: -kv[1])

In [97]:
N = torch.zeros((27,27), dtype = torch.int32)


In [61]:
#lookup table from characters to ints
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}

In [96]:
#2D array of bigram counts
log_likelihood = 0.0
for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        print(f'{ch1}{ch2}: {prob:4f}{logprob:4f}')

print(f'{log_likelihood=}')

.e: 0.047794-3.040846
em: 0.037654-3.279326
mm: 0.025294-3.677204
ma: 0.389943-0.941755
a.: 0.195957-1.629861
.o: 0.012300-4.398171
ol: 0.078019-2.550807
li: 0.177676-1.727794
iv: 0.015197-4.186665
vi: 0.354061-1.038285
ia: 0.138128-1.979576
a.: 0.195957-1.629861
.a: 0.137671-1.982892
av: 0.024613-3.704494
va: 0.249514-1.388240
a.: 0.195957-1.629861
log_likelihood=tensor(-38.7856)


In [73]:
#probability vector
p = N[0].float()
p = p/p.sum()

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [93]:
P = N.float()
P /= P.sum(1, keepdim=True)

In [92]:
#bigram name generator
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    ix = 0
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix==0:
            break
    print(''.join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.


## Neural network bigram model

In [99]:
import torch.nn.functional as F

In [141]:
#create training set of bigrams
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
ys

tensor([ 5, 13, 13,  1,  0])

In [142]:
xenc = F.one_hot(xs, num_classes=27).float() #input to network
yenc = F.one_hot(ys, num_classes=27).float()

In [143]:
xenc.shape

torch.Size([5, 27])

In [152]:
#initializing weights for 27 neurons, each with 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27), generator=g, requires_grad=True)

In [153]:
#forward pass
logits = xenc @ W #log counts (wx)
counts = logits.exp() #equivalent to count matrix
probs = counts/counts.sum(1, keepdims=True) #normalize counts to get prob distribution
loss = -probs[torch.arange(5), ys].log().mean()

In [154]:
#backward pass
W.grad = None
loss.backward()

In [155]:
W.data += -0.1 + W.grad

In [156]:
# create the dataset
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [157]:
# gradient descent
for k in range(1):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

3.7686190605163574


In [158]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

texzmkloglquszipczktxhkmpmzistttwinmlgdukzka.
zr.
rocxtpucjwtsc.
gmtokmxczisqytxugkwpt.
dajkkluydjmscdgu.
